# Homework 5: Monitoring

In [1]:
from starter import rag

In [3]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

It keeps a `while True` loop that:

1. calls the model,
2. checks the response for any `function_call`,
3. runs those tools and appends the outputs to the message history,
4. calls the model again with the updated history.

It stops when the model returns a response with no function calls, and the `has_function_calls` flag stays `False`, so the loop `break`s.


In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

In [ ]:

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

Overriding of current TracerProvider is not allowed


In [2]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [3]:
provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [4]:
from rag_helper import RAGBase


class RAGTraced(RAGBase):
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    
    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            results = super().search(query, num_results=num_results)
            span.set_attribute("search", "Complete.")
        return results

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            results = super().llm(prompt)
            span.set_attribute("input_tokens", results.usage.input_tokens)
            span.set_attribute("output_tokens", results.usage.output_tokens)
        return results

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            results = super().rag(query)
            span.set_attribute("rag", "Complete.")
        return results


In [5]:
from starter import index, client

In [ ]:


q = "How does the agentic loop keep calling the model until it stops?"
rag_traced = RAGTraced(index=index, llm_client=client)

result = rag_traced.rag(q)


{
    "name": "search",
    "context": {
        "trace_id": "0xb3b712a15adec6eefec222e019acc540",
        "span_id": "0xd159487b4dbed041",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xfd5b2f496d94c083",
    "start_time": "2026-07-15T00:56:30.485150Z",
    "end_time": "2026-07-15T00:56:30.486835Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "search": "Complete."
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "844d0b1f-f784-40d4-a672-47126a9cebe7",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xb3b712a15adec6eefec222e019acc540",
        "span_id": "0xc6540a78d04527f4",
        "trace_state"

# Question 1: First trace
## Answer: 3 spans

In [12]:
rag_traced = RAGTraced(index=index, llm_client=client)

result = rag_traced.rag(q)

{
    "name": "search",
    "context": {
        "trace_id": "0xaea5b73f85f5a545a036f9b822bd65fb",
        "span_id": "0xa780edbf3aa49198",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xe43246df09e92a40",
    "start_time": "2026-07-15T01:12:07.240356Z",
    "end_time": "2026-07-15T01:12:07.241325Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "search": "Complete."
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "844d0b1f-f784-40d4-a672-47126a9cebe7",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xaea5b73f85f5a545a036f9b822bd65fb",
        "span_id": "0x45fdadd47e0c8c19",
        "trace_state"

# Q2. Capturing metrics as span attributes

```
{
    "name": "llm",
    "context": {
        "trace_id": "0xaea5b73f85f5a545a036f9b822bd65fb",
        "span_id": "0x45fdadd47e0c8c19",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xe43246df09e92a40",
    "start_time": "2026-07-15T01:12:07.241693Z",
    "end_time": "2026-07-15T01:12:09.170069Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input_tokens": 7111,
        "output_tokens": 118
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "844d0b1f-f784-40d4-a672-47126a9cebe7",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
```

## Answer: 7000 input tokens

# Question 3: Span Timing


delta = 1784076990487 - 1784076992967 ~= 2480 ms

## Answer: > 2000 ms 

You might have to restart Jupyter using the sql exporter for OTel.


In [7]:
q = "How does the agentic loop keep calling the model until it stops?"
rag_traced = RAGTraced(index=index, llm_client=client)

result = rag_traced.rag(q)

In [8]:
# 1. Connect to database (creates file if it doesn't exist)
conn = sqlite3.connect('traces.db')
cursor = conn.cursor()


# 4. Query data
cursor.execute("SELECT * FROM spans")
#cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
rows = cursor.fetchall()
for row in rows:
    print(row)

# 5. Close connection
conn.close()

('search', 1784081663204326000, 1784081663207788000, None, None, None)
('llm', 1784081663208335000, 1784081665078125000, 7111, 96, None)
('rag', 1784081663204292000, 1784081665078709000, None, None, None)


# Question 4: Saving traces to SQLite

## Answer: rag, search, and llm

In [12]:
# 1. Connect to database (creates file if it doesn't exist)
conn = sqlite3.connect('traces.db')
cursor = conn.cursor()


# 4. Query data
cursor.execute("SELECT name, (end_time - start_time) as diff FROM spans WHERE name IN ('search', 'llm')")
#cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
rows = cursor.fetchall()
for row in rows:
    print(row)

# 5. Close connection
conn.close()

('search', 3462000)
('llm', 1869790000)


# Question 5: Querying trace data

## Answer: llm

In [13]:
result = rag_traced.rag(q)

In [14]:
result = rag_traced.rag(q)

In [15]:
result = rag_traced.rag(q)

In [ ]:
import pandas as pd

conn = sqlite3.connect('traces.db')
df = pd.read_sql_query("SELECT * FROM spans", conn)
conn.close()



In [19]:
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784081663204326000,1784081663207788000,NaN,NaN,None
1,llm,1784081663208335000,1784081665078125000,7111.0,96.0,None
2,rag,1784081663204292000,1784081665078709000,NaN,NaN,None
3,search,1784085863997536000,1784085864005736000,NaN,NaN,None
4,llm,1784085864006297000,1784085866699226000,7111.0,124.0,None
5,rag,1784085863997506000,1784085866700406000,NaN,NaN,None
6,search,1784085868249465000,1784085868250284000,NaN,NaN,None
7,llm,1784085868250922000,1784085869972614000,7111.0,122.0,None
8,rag,1784085868249448000,1784085869973261000,NaN,NaN,None
9,search,1784085870915350000,1784085870916199000,NaN,NaN,None


# Question 6: Token stability across runs

## Answer: They're identical